# 03 · Outcome Modeling (M0 → M3)

**Purpose:** Fit four xFG models on the field goal attempt data:

| Model | Description |
|-------|-------------|
| **M0** | Distance-only GLMM (no random effects) — baseline |
| **M1** | Full GLMM: distance B-splines + situational features + kicker:season + stadium random effects |
| **M2** | Augmented dataset: FG attempts + PATs + filtered non-attempts |
| **M3** | M1 formula + IPW weights (selection-bias corrected) |

**Inputs:**
- `data/fg_all.csv`
- `data/kicker_by_game.csv`
- `reports/attempt_pi/attempt_pi_oof_predictions_final.csv`

**Outputs:**
- `models/m0/m0_fg_B_logit.rds`
- `models/m1/m1_fg_B_logit.rds`
- `models/m2/m2_fg_B_aug_logit.rds`
- `models/m3/m3_fg_B_m1_logit_cf.rds`
- `reports/xfg_success/fg_full_with_predictions.csv`
- `reports/xfg_success/metrics_summary.csv`

In [1]:
# ============================================================
# 1. Parameters
# ============================================================
PROJECT_ROOT <- sub('[/\\][^/\\]*$', '', getwd())

data_dir     <- file.path(PROJECT_ROOT, 'data')
models_dir   <- file.path(PROJECT_ROOT, 'models')
reports_dir  <- file.path(PROJECT_ROOT, 'reports')

reports_out       <- file.path(reports_dir, 'xfg_success')
aug_data_dir      <- file.path(data_dir, 'augmented')
final_models_dir  <- file.path(models_dir, 'final_models')

# Canonical model artifact names
MODEL_FILE_M0     <- 'xfg_m0_dist_only_logit.rds'
MODEL_FILE_M1     <- 'xfg_m1_full_logit.rds'
MODEL_FILE_M2       <- 'xfg_m2_augmented_logit.rds'
MODEL_FILE_M2_NOPAT <- 'xfg_m2_augmented_nopat_logit.rds'
MODEL_FILE_M3       <- 'xfg_m3_ipw_logit.rds'
MODEL_FILE_M3_POP   <- 'xfg_m3_ipw_no_kicker_season_logit.rds'

# Canonical modeling/evaluation window
EVAL_SEASON_MIN <- 2015L
EVAL_SEASON_MAX <- 2025L

# Train/test split is sampled only within the canonical window
TEST_SEASONS       <- EVAL_SEASON_MIN:EVAL_SEASON_MAX
TEST_FRACTION      <- 0.20
TEST_GAME_IDS_FILE <- file.path(
  aug_data_dir,
  sprintf('test_fg_game_ids_%d_%d.csv', EVAL_SEASON_MIN, EVAL_SEASON_MAX)
)

# B-spline distance knots (outcome model)
DIST_KNOTS  <- c(28, 38, 48, 58)
DIST_BOUNDS <- c(18, 70)

# Weather main effects: natural splines on the standardized weather covariates.
# df = 3 is deliberately modest -- enough curvature to capture a physical
# non-linearity without spending degrees of freedom on noise.
WEATHER_VARS <- c('wind_z', 'temp_z', 'humidity_z')
WEATHER_DF   <- 3L

# M2 augmented data hyperparameters
PAT_WEIGHT         <- 0.10   # relative weight of PAT plays
NON_WEIGHT         <- 0.10   # relative weight of non-attempt plays
NON_PI_THRESHOLD   <- 0.25   # minimum propensity for non-attempt inclusion
NON_DIST_THRESHOLD <- 33     # minimum distance for non-attempt inclusion

# glmmTMB control
N_PARALLEL <- 6L

set.seed(20240517)
for (d in c(reports_out, aug_data_dir, final_models_dir)) {
  if (!dir.exists(d)) dir.create(d, recursive = TRUE, showWarnings = FALSE)
}
message('PROJECT_ROOT: ', PROJECT_ROOT)
message('final_models_dir: ', final_models_dir)

PROJECT_ROOT: X:/My Files/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model



final_models_dir: X:/My Files/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/models/final_models



In [2]:
# ============================================================
# 2. Imports
# ============================================================
dependencies <- c(
  'dplyr', 'tibble', 'tidyr', 'readr', 'stringr', 'purrr', 'ggplot2',
  'splines2', 'pROC', 'glmmTMB', 'Matrix'
)
installed <- rownames(installed.packages())
for (pkg in dependencies) {
  if (!pkg %in% installed) install.packages(pkg)
  suppressPackageStartupMessages(library(pkg, character.only = TRUE))
}
message('Libraries loaded.')

Libraries loaded.



In [3]:
# ============================================================
# 3. Helper Functions
# ============================================================

build_distance_bs <- function(x, knots = DIST_KNOTS, bknots = DIST_BOUNDS, degree = 3) {
  bs <- splines2::bSpline(x, knots = knots, Boundary.knots = bknots,
                           degree = degree, intercept = FALSE)
  bs <- as.data.frame(bs)
  names(bs) <- paste0('dist_bs_', seq_len(ncol(bs)))
  bs
}

# Natural-spline bases for the weather covariates (wind, temperature, humidity).
# Knots/boundary knots are derived ONCE from the reference set (attempted field
# goals) and then applied to every row via predict(), so that training rows,
# test rows, PAT rows, non-attempt rows and all prediction frames are mapped
# through an identical basis. Returns a data.frame of columns named
# <var>_ns_1 .. <var>_ns_WEATHER_DF.
# Boundary knots default to range(ref) -- fine now that 01_data_prep caps the
# one weather field (wind) that had an outlier at the source.
build_weather_ns <- function(df, ref_mask, vars = WEATHER_VARS, df_spline = WEATHER_DF) {
  out <- list()
  for (v in vars) {
    x   <- df[[v]]
    ref <- x[ref_mask & is.finite(x)]
    basis <- splines::ns(ref, df = df_spline)
    b <- predict(basis, x)                    # reuses the stored knots
    b <- as.data.frame(b)
    names(b) <- paste0(v, '_ns_', seq_len(ncol(b)))
    out[[v]] <- b
    message(sprintf('  %s: ns df=%d | knots=%s | boundary=%s',
                    v, df_spline,
                    paste(round(attr(basis, 'knots'), 3), collapse = ','),
                    paste(round(attr(basis, 'Boundary.knots'), 3), collapse = ',')))
  }
  dplyr::bind_cols(out)
}

brier   <- function(y, p) mean((y - p)^2, na.rm = TRUE)
logloss <- function(y, p, eps = 1e-15)
  -mean(y * log(pmin(pmax(p, eps), 1-eps)) + (1-y) * log(1-pmin(pmax(p, eps), 1-eps)), na.rm=TRUE)
auc_fn  <- function(y, p) tryCatch(as.numeric(pROC::auc(y, p, quiet = TRUE)), error = function(e) NA_real_)

calib_error <- function(y, p, nbins = 10) {
  bins <- ggplot2::cut_number(p, nbins)
  df <- tibble::tibble(y = y, p = p, bin = bins)
  res <- df %>% group_by(bin) %>%
    summarise(obs = mean(y), pred = mean(p), n = n(), .groups = 'drop')
  weighted.mean(abs(res$obs - res$pred), res$n)
}

calc_metrics <- function(y, p, set_name) {
  tibble::tibble(
    set        = set_name,
    n          = sum(!is.na(y) & !is.na(p)),
    brier      = brier(y, p),
    logloss    = logloss(y, p),
    auc        = auc_fn(y, p),
    calib_err  = tryCatch(calib_error(y, p), error = function(e) NA_real_),
    accuracy   = mean((p >= 0.5) == (y == 1), na.rm = TRUE),
    precision  = {
      tp <- sum(p >= 0.5 & y == 1, na.rm=TRUE)
      pp <- sum(p >= 0.5, na.rm=TRUE)
      if (pp == 0) NA_real_ else tp / pp
    },
    recall = {
      tp <- sum(p >= 0.5 & y == 1, na.rm=TRUE)
      ap <- sum(y == 1, na.rm=TRUE)
      if (ap == 0) NA_real_ else tp / ap
    }
  ) %>%
  mutate(f1 = if_else(!is.na(precision) & !is.na(recall) & (precision + recall) > 0,
                      2 * precision * recall / (precision + recall), NA_real_))
}

# Safe glmmTMB prediction (allows new RE levels)
predict_safe <- function(fit, newdata) {
  tryCatch(
    predict(fit, newdata = newdata, type = 'response', allow.new.levels = TRUE),
    error = function(e) {
      message('Prediction error: ', e$message)
      rep(NA_real_, nrow(newdata))
    }
  )
}

# Pool sparse random effect levels
pool_sparse_levels <- function(train_df, test_df, id_col, min_n = 5) {
  counts <- train_df %>% group_by(!!sym(id_col)) %>% summarise(n = n(), .groups = 'drop')
  valid  <- counts[[id_col]][counts$n >= min_n]
  train_out <- train_df %>% mutate(
    !!id_col := if_else(!!sym(id_col) %in% valid, !!sym(id_col), 'POOL'))
  test_out  <- test_df %>% mutate(
    !!id_col := if_else(!!sym(id_col) %in% valid, !!sym(id_col), 'POOL'))
  list(train = train_out, test = test_out)
}

# Effective sample size (Kish's formula)
ess <- function(w) {
  w <- w[is.finite(w)]
  if (!length(w)) return(NA_real_)
  (sum(w)^2) / sum(w^2)
}

# glmmTMB optimizer control
glmm_ctrl <- function(n_parallel = N_PARALLEL) {
  glmmTMB::glmmTMBControl(
    optimizer    = nlminb,
    optCtrl      = list(iter.max = 1000, eval.max = 1000),
    parallel     = n_parallel
  )
}


## 4. Load Data

In [4]:
fg_all          <- readr::read_csv(file.path(data_dir, 'fg_all.csv'),       show_col_types = FALSE)
kicker_by_game  <- readr::read_csv(file.path(data_dir, 'kicker_by_game.csv'), show_col_types = FALSE)
preds_oof       <- readr::read_csv(file.path(reports_dir, 'attempt_pi',
                     'attempt_pi_oof_predictions_final.csv'), show_col_types = FALSE)

# A non-attempt's kick_distance is hypothetical (yardline_100 + 17) and runs to
# 116 yards. No model in this notebook can use such a row: the outcome model's
# B-spline is only defined on [18, 70] and would have to extrapolate. Drop them
# at load so the basis is never evaluated outside its support - previously this
# produced basis values near +/-183 and an ill-conditioning warning from
# splines2, and those rows went on to form 91% of M2's pseudo-miss block.
n_before <- nrow(fg_all)
fg_all <- fg_all %>%
  filter(!(attempted == 0L & is_pat == 0L & (is.na(kick_distance) | kick_distance > 70)))
message('Dropped ', n_before - nrow(fg_all),
        ' non-attempt rows outside kickable range (hypothetical distance > 70)')

message('fg_all:         ', nrow(fg_all), ' rows')
message('kicker_by_game: ', nrow(kicker_by_game), ' rows')
message('preds_oof:      ', nrow(preds_oof), ' rows | seasons ',
        min(preds_oof$season), '-', max(preds_oof$season))

Dropped 51509 non-attempt rows outside kickable range (hypothetical distance > 70)



fg_all:         86159 rows



kicker_by_game: 12184 rows



preds_oof:      44395 rows | seasons 2015-2025



In [5]:
# ============================================================
# 5. Feature Engineering & IPW Join
# ============================================================

# Distance B-splines
bs_mat <- build_distance_bs(fg_all$kick_distance)

# Natural-spline bases for the weather main effects (reviewer: the weather
# relationships are plausibly non-linear on physical grounds). Knots are fixed
# once from the attempted-kick distribution and then applied to every row, so
# FG / PAT / non-attempt rows and all newdata frames share an identical basis.
# This mirrors how the distance basis is handled and avoids relying on formula
# -embedded ns() to re-derive knots at predict time.
wx_mat <- build_weather_ns(fg_all, ref_mask = (fg_all$attempted == 1L & fg_all$is_pat == 0L))

fg_base <- dplyr::bind_cols(fg_all, bs_mat, wx_mat) %>%
  mutate(
    kick_made    = if_else(attempted == 1L & !is.na(kick_result),
                           as.integer(kick_result == 'made'), NA_integer_),
    season_f     = as.character(season),
    kicker_player_id = as.character(kicker_player_id),
    stadium_id   = as.character(stadium_id),
    # Re-label indoors as factor for glmmTMB
    indoors      = dplyr::coalesce(as.integer(indoors), 0L),
    is_turf      = dplyr::coalesce(as.integer(is_turf), 0L),
    high_altitude = dplyr::coalesce(as.integer(high_altitude), 0L)
  ) %>%
  filter(season >= EVAL_SEASON_MIN, season <= EVAL_SEASON_MAX)

# Restrict OOF weights to the same canonical window for consistency
preds_oof <- preds_oof %>%
  filter(season >= EVAL_SEASON_MIN, season <= EVAL_SEASON_MAX)

# Join kicker_by_game to fill kicker IDs for non-attempts
fg_base <- fg_base %>%
  left_join(kicker_by_game, by = c('game_id', 'posteam' = 'team'), suffix = c('', '_game')) %>%
  mutate(
    kicker_player_id = dplyr::coalesce(kicker_player_id, kicker_player_id_game)
  ) %>%
  select(-kicker_player_id_game)

# Join OOF propensity weights (for FG attempts only)
oof_join <- preds_oof %>%
  mutate(game_id = as.character(game_id), play_id = as.character(play_id)) %>%
  select(game_id, play_id, p_hat_attempt_clipped, w_ipw_final, weight_multinom_hajek)

fg_base <- fg_base %>%
  mutate(game_id = as.character(game_id), play_id = as.character(play_id)) %>%
  left_join(oof_join, by = c('game_id', 'play_id'))

# PATs are not fourth-down decisions, so the propensity model never scores them.
# They are assigned the extreme values of the observed FG range: a PAT is the
# situation a coach is most certain to kick from, hence the maximum propensity
# and the minimum IPW weight.
max_p_hat_fg    <- max(fg_base$p_hat_attempt_clipped[fg_base$is_pat == 0L], na.rm = TRUE)
min_ipw_w_fg    <- min(fg_base$w_ipw_final[fg_base$is_pat == 0L], na.rm = TRUE)

# Rows the propensity model did not score get NA, not a constant.
#
# This previously fell back to `fg_prev`, the marginal FG attempt prevalence
# (0.254118). Because the propensity frame was truncated at kick_distance <= 70,
# two thirds of the non-attempts were unscored, and that constant cleared M2's
# 0.25 propensity floor by 0.004118 - so 21,524 plays at 71-116 hypothetical
# yards entered M2 as certain misses on the strength of a rounding accident.
# The propensity model is now fitted across the full field, so every genuine
# fourth-down decision carries a real estimate and the fallback is unnecessary.
# Anything still unscored (qb_kneel, missing clock) is not a decision and is
# excluded by the !is.na() screens downstream.
fg_base <- fg_base %>%
  mutate(
    p_hat_multinom = dplyr::case_when(
      is_pat == 1L                  ~ as.numeric(max_p_hat_fg),
      !is.na(p_hat_attempt_clipped) ~ as.numeric(p_hat_attempt_clipped),
      TRUE                          ~ NA_real_
    ),
    # Canonical case weight for M3: the clipped, stabilized, Hajek-normalized,
    # 3-sigma-capped weight from notebook 02. M3 previously *filtered* on
    # w_ipw_final but was *fitted* with weight_multinom_hajek, which skipped the
    # cap entirely and reached 1632 on a single attempt.
    w_fit = dplyr::case_when(
      is_pat == 1L        ~ as.numeric(min_ipw_w_fg),
      !is.na(w_ipw_final) ~ as.numeric(w_ipw_final),
      TRUE                ~ NA_real_
    )
  )

n_scored   <- sum(!is.na(fg_base$p_hat_multinom) & fg_base$is_pat == 0L & fg_base$attempted == 0L)
n_unscored <- sum(is.na(fg_base$p_hat_multinom)  & fg_base$is_pat == 0L & fg_base$attempted == 0L)
message('Non-attempt propensity coverage: ', n_scored, ' scored / ', n_unscored, ' unscored')

message('fg_base built: ', nrow(fg_base), ' rows | seasons ',
        min(fg_base$season), '-', max(fg_base$season))

Warning message in splines2::bSpline(x, knots = knots, Boundary.knots = bknots, :
"Some 'x' values beyond boundary knots may cause ill-conditioned basis
functions."


  wind_z: ns df=3 | knots=-0.542,0.348 | boundary=-1.076,6.045



  temp_z: ns df=3 | knots=-0.296,0.601 | boundary=-4.331,3.035



  humidity_z: ns df=3 | knots=-0.546,0.451 | boundary=-3.012,2.236



Non-attempt propensity coverage: 11115 scored / 27 unscored



fg_base built: 36979 rows | seasons 2015-2025



In [6]:
# ============================================================
# 6. Train / Test Split
# ============================================================
# Persistent test game IDs for reproducibility across runs.

if (file.exists(TEST_GAME_IDS_FILE)) {
  test_ids_df <- readr::read_csv(TEST_GAME_IDS_FILE, show_col_types = FALSE)
  test_game_ids <- test_ids_df$game_id
  message('Loaded existing test game IDs: ', length(test_game_ids))
} else {
  candidate_games <- fg_base %>%
    filter(is_pat == 0L, attempted == 1L, season %in% TEST_SEASONS) %>%
    distinct(game_id, season)

  test_game_ids <- candidate_games %>%
    group_by(season) %>%
    slice_sample(prop = TEST_FRACTION) %>%
    ungroup() %>%
    pull(game_id)

  readr::write_csv(tibble::tibble(game_id = test_game_ids), TEST_GAME_IDS_FILE)
  message('Created test game IDs: ', length(test_game_ids))
}

# Subset master datasets within canonical evaluation window
# Blocked FGs are excluded: they are not outcomes of kicker skill and
# should not be treated as misses in the success model.
FG_master  <- fg_base %>%
  filter(is_pat == 0L, attempted == 1L, !is.na(kick_made),
         kick_result != 'blocked',
         season >= EVAL_SEASON_MIN, season <= EVAL_SEASON_MAX)
PAT_master <- fg_base %>%
  filter(is_pat == 1L,
         season >= EVAL_SEASON_MIN, season <= EVAL_SEASON_MAX)
# Non-attempts carry a *hypothetical* kick distance (yardline_100 + 17), which
# runs to 116 yards. The outcome model's B-spline has boundary knots at 18/70,
# so anything beyond 70 is extrapolated - basis values reached +/-183 and R
# warned about ill-conditioning. Cap the pool at the basis boundary so no row
# the outcome model ever sees sits outside its support.
NON_master <- fg_base %>%
  filter(attempted == 0L, is_pat == 0L,
         !is.na(kick_distance), kick_distance <= 70,
         season >= EVAL_SEASON_MIN, season <= EVAL_SEASON_MAX)

train_df <- FG_master %>% filter(!game_id %in% test_game_ids)
test_df  <- FG_master %>% filter(game_id  %in% test_game_ids)

train_game_ids <- unique(train_df$game_id)

message(sprintf('FG master: %d | Train: %d | Test: %d',
  nrow(FG_master), nrow(train_df), nrow(test_df)))
message('PAT: ', nrow(PAT_master), ' | Non-attempts: ', nrow(NON_master))
message('Canonical window: ', EVAL_SEASON_MIN, '-', EVAL_SEASON_MAX)
message('kick_result distribution in FG_master:')
print(table(FG_master$kick_result))

Loaded existing test game IDs: 593



FG master: 11548 | Train: 9297 | Test: 2251



PAT: 14073 | Non-attempts: 11142



Canonical window: 2015-2025



kick_result distribution in FG_master:




  made missed 
  9962   1586 


In [7]:
# ============================================================
# 7. Model Formulas
# ============================================================
n_bs <- sum(startsWith(names(bs_mat), 'dist_bs_'))
bs_terms <- paste0('dist_bs_', seq_len(n_bs))

# Natural-spline main-effect terms for wind, temperature and humidity.
weather_ns_terms <- unlist(lapply(WEATHER_VARS,
                                  function(v) paste0(v, '_ns_', seq_len(WEATHER_DF))))

# M0: distance-only, no random effects
form_m0 <- as.formula(paste('kick_made ~', paste(bs_terms, collapse = ' + ')))

# M1: full fixed + random effects
# Weather specification (reviewer: "what is the relationship between the weather
# variables and probability of success? ... these might not be [linear] based on
# physics formulas", plus "I would guess that humidity would impact field goal
# success rate"):
# - wind, temperature and humidity now enter as natural-spline MAIN EFFECTS
#   (df = 3 each) instead of linear terms
# - humidity_z is new to the model
# - the previous wind_z:temp_z term and the wind_z x distance-spline interaction
#   block are DROPPED. This is a deliberate simplification: weather is assumed to
#   act additively on the logit rather than to amplify with distance. The
#   interaction-rich alternative was considered and rejected as overkill for the
#   sample size available in the long-distance tail.
# - rain/snow stay as binary flags; the remaining context flags are unchanged
fixed_terms_m1 <- c(
  bs_terms,
  weather_ns_terms,
  'is_turf', 'is_snow_sleet', 'is_rain_showers',
  'clock_running', 'iced', 'l2m', 'clock_running:l2m',
  'playoffs'
)
re_terms_m1 <- c(
  '(1 | stadium_id)',
  '(1 | kicker_player_id:season_f)'
)
form_m1 <- as.formula(paste(
  'kick_made ~',
  paste(c(fixed_terms_m1, re_terms_m1), collapse = ' + ')
))

# M3-pop: M1 structure without kicker:season random effect
re_terms_m3_pop <- c('(1 | stadium_id)')
form_m3_pop <- as.formula(paste(
  'kick_made ~',
  paste(c(fixed_terms_m1, re_terms_m3_pop), collapse = ' + ')
))

message('M0 terms: ', length(bs_terms))
message('M1 fixed terms: ', length(fixed_terms_m1))
message('Formulas built.')

M0 terms: 7



M1 fixed terms: 24



Formulas built.



## 8. Fit Models

In [8]:
# ============================================================
# M0: Distance-only GLMM (no random effects)
# ============================================================
message('Fitting M0 ...')
m0 <- glmmTMB::glmmTMB(
  formula   = form_m0,
  data      = train_df,
  family    = binomial(link = 'logit'),
  control   = glmm_ctrl()
)

saveRDS(m0, file.path(final_models_dir, MODEL_FILE_M0))
message('M0 converged: ', !isTRUE(m0$fit$convergence != 0))
message('M0 AIC: ', round(AIC(m0), 1))
cat('\n=== M0 Formula ===\n'); print(formula(m0))
cat('\n=== M0 Summary ===\n'); print(summary(m0))


Fitting M0 ...



M0 converged: TRUE



M0 AIC: 6345.7




=== M0 Formula ===


kick_made ~ dist_bs_1 + dist_bs_2 + dist_bs_3 + dist_bs_4 + dist_bs_5 + 
    dist_bs_6 + dist_bs_7



=== M0 Summary ===


 Family: binomial  ( logit )
Formula:          
kick_made ~ dist_bs_1 + dist_bs_2 + dist_bs_3 + dist_bs_4 + dist_bs_5 +  
    dist_bs_6 + dist_bs_7
Data: train_df

      AIC       BIC    logLik -2*log(L)  df.resid 
   6345.7    6402.8   -3164.9    6329.7      9289 


Conditional model:
            Estimate Std. Error z value Pr(>|z|)   
(Intercept)   13.099      4.483   2.922  0.00348 **
dist_bs_1     -9.366      5.090  -1.840  0.06575 . 
dist_bs_2     -9.068      4.334  -2.092  0.03640 * 
dist_bs_3    -10.927      4.538  -2.408  0.01606 * 
dist_bs_4    -12.218      4.457  -2.741  0.00612 **
dist_bs_5    -12.333      4.524  -2.726  0.00641 **
dist_bs_6    -14.145      4.484  -3.155  0.00161 **
dist_bs_7    -15.032      4.781  -3.144  0.00167 **
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1


In [9]:
# ============================================================
# M1: Full GLMM (no weights)
# ============================================================
message('Fitting M1 ...')

# Pool sparse RE levels
pool_kicker  <- pool_sparse_levels(train_df, test_df, 'kicker_player_id', min_n = 5)
pool_stadium <- pool_sparse_levels(pool_kicker$train, pool_kicker$test, 'stadium_id', min_n = 5)
train_m1 <- pool_stadium$train
test_m1  <- pool_stadium$test

m1 <- glmmTMB::glmmTMB(
  formula = form_m1,
  data    = train_m1,
  family  = binomial(link = 'logit'),
  control = glmm_ctrl()
)

saveRDS(m1, file.path(final_models_dir, MODEL_FILE_M1))
message('M1 converged: ', !isTRUE(m1$fit$convergence != 0))
message('M1 AIC: ', round(AIC(m1), 1))
cat('\n=== M1 Formula ===\n'); print(formula(m1))
cat('\n=== M1 Summary ===\n'); print(summary(m1))


Fitting M1 ...



M1 converged: TRUE



M1 AIC: 6333.8




=== M1 Formula ===


kick_made ~ dist_bs_1 + dist_bs_2 + dist_bs_3 + dist_bs_4 + dist_bs_5 + 
    dist_bs_6 + dist_bs_7 + wind_z_ns_1 + wind_z_ns_2 + wind_z_ns_3 + 
    temp_z_ns_1 + temp_z_ns_2 + temp_z_ns_3 + humidity_z_ns_1 + 
    humidity_z_ns_2 + humidity_z_ns_3 + is_turf + is_snow_sleet + 
    is_rain_showers + clock_running + iced + l2m + clock_running:l2m + 
    playoffs + (1 | stadium_id) + (1 | kicker_player_id:season_f)



=== M1 Summary ===


 Family: binomial  ( logit )
Formula:          
kick_made ~ dist_bs_1 + dist_bs_2 + dist_bs_3 + dist_bs_4 + dist_bs_5 +  
    dist_bs_6 + dist_bs_7 + wind_z_ns_1 + wind_z_ns_2 + wind_z_ns_3 +  
    temp_z_ns_1 + temp_z_ns_2 + temp_z_ns_3 + humidity_z_ns_1 +  
    humidity_z_ns_2 + humidity_z_ns_3 + is_turf + is_snow_sleet +  
    is_rain_showers + clock_running + iced + l2m + clock_running:l2m +  
    playoffs + (1 | stadium_id) + (1 | kicker_player_id:season_f)
Data: train_m1

      AIC       BIC    logLik -2*log(L)  df.resid 
   6333.8    6526.5   -3139.9    6279.8      9270 

Random effects:

Conditional model:
 Groups                    Name        Variance Std.Dev.
 stadium_id                (Intercept) 0.009547 0.09771 
 kicker_player_id:season_f (Intercept) 0.083279 0.28858 
Number of obs: 9297, groups:  stadium_id, 43; kicker_player_id:season_f, 444

Conditional model:
                   Estimate Std. Error z value Pr(>|z|)   
(Intercept)        13.66908    4.60566   2.968  0.0

In [10]:
# ============================================================
# M2: Augmented dataset (FG + PAT + filtered non-attempts)
# ============================================================
message('Fitting M2 ...')

# Filter non-attempts: pi >= threshold, distance > threshold
non_filtered <- NON_master %>%
  filter(
    !is.na(p_hat_multinom) & p_hat_multinom >= NON_PI_THRESHOLD,
    !is.na(kick_distance)  & kick_distance  >  NON_DIST_THRESHOLD,
    kick_distance <= 70
  ) %>%
  mutate(
    kick_made  = 0L,
    aug_weight = NON_WEIGHT
  )

# The propensity floor must now operate on real estimates, not a constant.
stopifnot(all(!is.na(non_filtered$p_hat_multinom)))
stopifnot(max(non_filtered$kick_distance) <= 70)
message('M2 non-attempt pool: ', nrow(non_filtered), ' rows')
cat('  hypothetical kick distance:
'); print(summary(non_filtered$kick_distance))
cat('  propensity of included rows:
'); print(summary(non_filtered$p_hat_multinom))

pat_subset <- PAT_master %>%
  mutate(
    kick_made  = if_else(!is.na(kick_result), as.integer(kick_result == 'made'), NA_integer_),
    aug_weight = PAT_WEIGHT
  ) %>%
  filter(!is.na(kick_made))

# Row-bind FG train + PAT + non-attempts (RE pooling applied uniformly below)
train_m2 <- dplyr::bind_rows(
  train_m1 %>% mutate(aug_weight = 1.0),
  pat_subset %>% filter(game_id %in% train_game_ids),
  non_filtered %>% filter(game_id %in% train_game_ids)
)

# Pool RE levels consistently using FG train-set counts
all_valid_kickers  <- names(which(table(train_m1$kicker_player_id) >= 5))
all_valid_stadiums <- names(which(table(train_m1$stadium_id) >= 5))

train_m2 <- train_m2 %>%
  mutate(
    kicker_player_id = if_else(kicker_player_id %in% all_valid_kickers,
                               kicker_player_id, 'POOL'),
    stadium_id = if_else(stadium_id %in% all_valid_stadiums,
                         stadium_id, 'POOL')
  ) %>%
  filter(!is.na(kick_made), !is.na(aug_weight), aug_weight > 0)

m2 <- glmmTMB::glmmTMB(
  formula = form_m1,
  data    = train_m2,
  weights = aug_weight,
  family  = binomial(link = 'logit'),
  control = glmm_ctrl()
)

saveRDS(m2, file.path(final_models_dir, MODEL_FILE_M2))
message('M2 converged: ', !isTRUE(m2$fit$convergence != 0))
message('M2 AIC: ', round(AIC(m2), 1))
message('M2 train rows: ', nrow(train_m2),
        ' (FG: ', nrow(train_m1), ', PAT: ', nrow(filter(pat_subset, game_id %in% train_game_ids)),
        ', NON: ', nrow(filter(non_filtered, game_id %in% train_game_ids)), ')')
cat('\n=== M2 Formula ===\n'); print(formula(m2))
cat('\n=== M2 Summary ===\n'); print(summary(m2))


Fitting M2 ...



M2 non-attempt pool: 1923 rows



  hypothetical kick distance:


   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  34.00   41.00   48.00   47.06   53.00   67.00 


  propensity of included rows:


...

M2 train rows: 21740 (FG: 9297, PAT: 10955, NON: 1488)




=== M2 Formula ===


kick_made ~ dist_bs_1 + dist_bs_2 + dist_bs_3 + dist_bs_4 + dist_bs_5 + 
    dist_bs_6 + dist_bs_7 + wind_z_ns_1 + wind_z_ns_2 + wind_z_ns_3 + 
    temp_z_ns_1 + temp_z_ns_2 + temp_z_ns_3 + humidity_z_ns_1 + 
    humidity_z_ns_2 + humidity_z_ns_3 + is_turf + is_snow_sleet + 
    is_rain_showers + clock_running + iced + l2m + clock_running:l2m + 
    playoffs + (1 | stadium_id) + (1 | kicker_player_id:season_f)



=== M2 Summary ===


 Family: binomial  ( logit )
Formula:          
kick_made ~ dist_bs_1 + dist_bs_2 + dist_bs_3 + dist_bs_4 + dist_bs_5 +  
    dist_bs_6 + dist_bs_7 + wind_z_ns_1 + wind_z_ns_2 + wind_z_ns_3 +  
    temp_z_ns_1 + temp_z_ns_2 + temp_z_ns_3 + humidity_z_ns_1 +  
    humidity_z_ns_2 + humidity_z_ns_3 + is_turf + is_snow_sleet +  
    is_rain_showers + clock_running + iced + l2m + clock_running:l2m +  
    playoffs + (1 | stadium_id) + (1 | kicker_player_id:season_f)
Data: train_m2
Weights: aug_weight

      AIC       BIC    logLik -2*log(L)  df.resid 
   7270.7    7486.4   -3608.4    7216.7     21713 

Random effects:

Conditional model:
 Groups                    Name        Variance Std.Dev.
 stadium_id                (Intercept) 0.003503 0.05919 
 kicker_player_id:season_f (Intercept) 0.059358 0.24363 
Number of obs: 21740, groups:  stadium_id, 43; kicker_player_id:season_f, 450

Conditional model:
                   Estimate Std. Error z value Pr(>|z|)   
(Intercept)        12.29910   

In [11]:
# ============================================================
# M2 variant: non-attempt pseudo-misses ONLY (no PAT augmentation)
# ============================================================
# Reviewer Main-4a asked what the PAT rows actually buy us. This variant
# isolates that: identical training set and weights, minus the PAT block, so
# the difference between m2 and m2_nopat is attributable to the PATs alone.
message('Fitting M2 (no PAT) ...')

train_m2_nopat <- dplyr::bind_rows(
  train_m1 %>% mutate(aug_weight = 1.0),
  non_filtered %>% filter(game_id %in% train_game_ids)
) %>%
  mutate(
    kicker_player_id = if_else(kicker_player_id %in% all_valid_kickers,
                               kicker_player_id, 'POOL'),
    stadium_id = if_else(stadium_id %in% all_valid_stadiums,
                         stadium_id, 'POOL')
  ) %>%
  filter(!is.na(kick_made), !is.na(aug_weight), aug_weight > 0)

m2_nopat <- glmmTMB::glmmTMB(
  formula = form_m1,
  data    = train_m2_nopat,
  weights = aug_weight,
  family  = binomial(link = 'logit'),
  control = glmm_ctrl()
)

saveRDS(m2_nopat, file.path(final_models_dir, MODEL_FILE_M2_NOPAT))
message('M2 (no PAT) converged: ', !isTRUE(m2_nopat$fit$convergence != 0))
message('M2 (no PAT) AIC: ', round(AIC(m2_nopat), 1))
message('M2 (no PAT) train rows: ', nrow(train_m2_nopat),
        ' (FG: ', nrow(train_m1),
        ', NON: ', nrow(filter(non_filtered, game_id %in% train_game_ids)),
        ', PAT: 0)')


Fitting M2 (no PAT) ...



M2 (no PAT) converged: TRUE



M2 (no PAT) AIC: 6796.4



M2 (no PAT) train rows: 10785 (FG: 9297, NON: 1488, PAT: 0)



In [12]:
# ============================================================
# M3: M1 formula + IPW weights (counterfactual model)
# ============================================================
message('Fitting M3 ...')

# Use train_m1 with w_ipw_final as case weight
# Filter and weight on the SAME column. These previously disagreed: the filter
# used w_ipw_final (3-sigma capped) while the fit used weight_multinom_hajek
# (uncapped, max 1632), so the model was weighted by a vector the paper did not
# describe and a single kick outweighed ~2,700 median-weight ones.
train_m3 <- train_m1 %>%
  filter(!is.na(w_fit), is.finite(w_fit), w_fit > 0)

message('M3 case weights: min=', round(min(train_m3$w_fit), 3),
        ' median=', round(median(train_m3$w_fit), 3),
        ' max=', round(max(train_m3$w_fit), 3))
stopifnot(max(train_m3$w_fit) < 50)

m3 <- glmmTMB::glmmTMB(
  formula = form_m1,
  data    = train_m3,
  weights = w_fit,
  family  = binomial(link = 'logit'),
  control = glmm_ctrl()
)

saveRDS(m3, file.path(final_models_dir, MODEL_FILE_M3))
message('M3 converged: ', !isTRUE(m3$fit$convergence != 0))
message('M3 AIC: ', round(AIC(m3), 1))
cat('\n=== M3 Formula ===\n'); print(formula(m3))
cat('\n=== M3 Summary ===\n'); print(summary(m3))


Fitting M3 ...



M3 case weights: min=0.628 median=0.713 max=8.493



Warning message in eval(family$initialize):
"non-integer #successes in a binomial glm!"


M3 converged: TRUE



M3 AIC: 6678




=== M3 Formula ===


kick_made ~ dist_bs_1 + dist_bs_2 + dist_bs_3 + dist_bs_4 + dist_bs_5 + 
    dist_bs_6 + dist_bs_7 + wind_z_ns_1 + wind_z_ns_2 + wind_z_ns_3 + 
    temp_z_ns_1 + temp_z_ns_2 + temp_z_ns_3 + humidity_z_ns_1 + 
    humidity_z_ns_2 + humidity_z_ns_3 + is_turf + is_snow_sleet + 
    is_rain_showers + clock_running + iced + l2m + clock_running:l2m + 
    playoffs + (1 | stadium_id) + (1 | kicker_player_id:season_f)



=== M3 Summary ===


 Family: binomial  ( logit )
Formula:          
kick_made ~ dist_bs_1 + dist_bs_2 + dist_bs_3 + dist_bs_4 + dist_bs_5 +  
    dist_bs_6 + dist_bs_7 + wind_z_ns_1 + wind_z_ns_2 + wind_z_ns_3 +  
    temp_z_ns_1 + temp_z_ns_2 + temp_z_ns_3 + humidity_z_ns_1 +  
    humidity_z_ns_2 + humidity_z_ns_3 + is_turf + is_snow_sleet +  
    is_rain_showers + clock_running + iced + l2m + clock_running:l2m +  
    playoffs + (1 | stadium_id) + (1 | kicker_player_id:season_f)
Data: train_m3
Weights: w_fit

      AIC       BIC    logLik -2*log(L)  df.resid 
   6678.0    6870.7   -3312.0    6624.0      9270 

Random effects:

Conditional model:
 Groups                    Name        Variance Std.Dev.
 stadium_id                (Intercept) 0.02696  0.1642  
 kicker_player_id:season_f (Intercept) 0.43190  0.6572  
Number of obs: 9297, groups:  stadium_id, 43; kicker_player_id:season_f, 444

Conditional model:
                    Estimate Std. Error z value Pr(>|z|)    
(Intercept)        12.504505   3.8

In [13]:
# ============================================================
# M3-pop: IPW model without kicker:season random effect
# ============================================================
message('Fitting M3-pop (no kicker:season RE) ...')

m3_pop <- glmmTMB::glmmTMB(
  formula = form_m3_pop,
  data    = train_m3,
  weights = w_fit,
  family  = binomial(link = 'logit'),
  control = glmm_ctrl()
)

saveRDS(m3_pop, file.path(final_models_dir, MODEL_FILE_M3_POP))
message('M3-pop converged: ', !isTRUE(m3_pop$fit$convergence != 0))
message('M3-pop AIC: ', round(AIC(m3_pop), 1))
cat('\n=== M3-pop Formula ===\n'); print(formula(m3_pop))
cat('\n=== M3-pop Summary ===\n'); print(summary(m3_pop))


Fitting M3-pop (no kicker:season RE) ...



Warning message in eval(family$initialize):
"non-integer #successes in a binomial glm!"


M3-pop converged: TRUE



M3-pop AIC: 6806.3




=== M3-pop Formula ===


kick_made ~ dist_bs_1 + dist_bs_2 + dist_bs_3 + dist_bs_4 + dist_bs_5 + 
    dist_bs_6 + dist_bs_7 + wind_z_ns_1 + wind_z_ns_2 + wind_z_ns_3 + 
    temp_z_ns_1 + temp_z_ns_2 + temp_z_ns_3 + humidity_z_ns_1 + 
    humidity_z_ns_2 + humidity_z_ns_3 + is_turf + is_snow_sleet + 
    is_rain_showers + clock_running + iced + l2m + clock_running:l2m + 
    playoffs + (1 | stadium_id)



=== M3-pop Summary ===


 Family: binomial  ( logit )
Formula:          
kick_made ~ dist_bs_1 + dist_bs_2 + dist_bs_3 + dist_bs_4 + dist_bs_5 +  
    dist_bs_6 + dist_bs_7 + wind_z_ns_1 + wind_z_ns_2 + wind_z_ns_3 +  
    temp_z_ns_1 + temp_z_ns_2 + temp_z_ns_3 + humidity_z_ns_1 +  
    humidity_z_ns_2 + humidity_z_ns_3 + is_turf + is_snow_sleet +  
    is_rain_showers + clock_running + iced + l2m + clock_running:l2m +  
    playoffs + (1 | stadium_id)
Data: train_m3
Weights: w_fit

      AIC       BIC    logLik -2*log(L)  df.resid 
   6806.3    6991.8   -3377.1    6754.3      9271 

Random effects:

Conditional model:
 Groups     Name        Variance Std.Dev.
 stadium_id (Intercept) 0.04192  0.2047  
Number of obs: 9297, groups:  stadium_id, 43

Conditional model:
                    Estimate Std. Error z value Pr(>|z|)    
(Intercept)        12.179102   3.709507   3.283 0.001026 ** 
dist_bs_1          -8.858665   4.321245  -2.050 0.040362 *  
dist_bs_2          -7.699126   3.510078  -2.193 0.028276 *  
dist

In [14]:
# ============================================================
# M3-pop variant B: marginalize the kicker RE out of M3 (no refit)
# ============================================================
# Reviewer Main-5: "Instead of using a new model M2-pop, it seems like you could
# use M2 while setting kicker coefficients to zero to achieve the same result."
# (Quoted verbatim; the reviewer wrote before the relabel, so the IPW model is
# M3 and the population baseline M3-pop in the numbering used below.)
#
# glmmTMB's predict() only supports re.form = NULL (condition on all REs) or
# re.form = NA (condition on none) -- it has no partial re.form. We need the
# kicker:season intercept set to zero while KEEPING the stadium intercept, so we
# reconstruct that prediction explicitly on the link scale:
#
#     eta_pop = eta(re.form = NA) + b_stadium[stadium_id]
#
# which is exact for a model whose only REs are independent random intercepts.
#
# Naming note: this sets the kicker random intercept to its mean of zero. For a
# logit link that yields the conditional-mode ("typical kicker") prediction, not
# a true integral over the kicker RE distribution -- those differ by Jensen's
# inequality. We report it as the reviewer's zeroed-kicker baseline.
predict_kicker_zeroed <- function(fit, newdata, stadium_col = 'stadium_id') {
  eta_fixed <- predict(fit, newdata = as.data.frame(newdata), type = 'link',
                       re.form = NA, allow.new.levels = TRUE)
  re <- glmmTMB::ranef(fit)$cond
  stad_re <- re[[stadium_col]]
  b <- setNames(stad_re[[1]], rownames(stad_re))
  add <- unname(b[as.character(newdata[[stadium_col]])])
  add[is.na(add)] <- 0                     # unseen venue -> population mean
  stats::plogis(eta_fixed + add)
}

message('Building M3-pop variant B (kicker RE zeroed out of M3) ...')

# Sanity: variant B must equal M3 for a kicker whose BLUP is ~0, and must not
# depend on kicker identity at all.
chk <- test_m1[1:min(200, nrow(test_m1)), , drop = FALSE]
chk_alt <- chk %>% mutate(kicker_player_id = 'POOL')
pb1 <- predict_kicker_zeroed(m3, chk)
pb2 <- predict_kicker_zeroed(m3, chk_alt)
message('  variant B kicker-invariance check | max |diff| = ',
        signif(max(abs(pb1 - pb2), na.rm = TRUE), 3), ' (should be ~0)')
message('  variant B mean p = ', round(mean(pb1, na.rm = TRUE), 4),
        ' | M3 mean p = ', round(mean(predict_safe(m3, chk), na.rm = TRUE), 4))


Building M3-pop variant B (kicker RE zeroed out of M3) ...



  variant B kicker-invariance check | max |diff| = 0 (should be ~0)



  variant B mean p = 0.8586 | M3 mean p = 0.8481



## 9. In-Sample Evaluation (Full Data)

Re-fit each model on **all** FG attempts (`FG_master`) to quantify explanatory power.
These fits are diagnostic only — canonical train-set models remain in `models/`.


In [15]:
# ============================================================
# 9a. In-Sample Model Fits (Full Data)
# ============================================================
# Purpose: quantify in-sample explanatory power (how much of FG kicking can we explain?)
# Diagnostic only — not saved as .rds canonical models.
message('Building in-sample pooled FG_master ...')

FG_master_pooled <- FG_master %>%
  mutate(
    kicker_player_id = if_else(kicker_player_id %in% all_valid_kickers,
                               kicker_player_id, 'POOL'),
    stadium_id = if_else(stadium_id %in% all_valid_stadiums,
                         stadium_id, 'POOL')
  )

message('Fitting M0_full ...')
m0_full <- glmmTMB::glmmTMB(
  formula = form_m0,
  data    = FG_master_pooled,
  family  = binomial(link = 'logit'),
  control = glmm_ctrl()
)
message('M0_full AIC: ', round(AIC(m0_full), 1))

message('Fitting M1_full ...')
m1_full <- glmmTMB::glmmTMB(
  formula = form_m1,
  data    = FG_master_pooled,
  family  = binomial(link = 'logit'),
  control = glmm_ctrl()
)
message('M1_full AIC: ', round(AIC(m1_full), 1))
cat('\n=== M1_full Summary ===\n'); print(summary(m1_full))

message('Fitting M3_full ...')
m3_full_data <- FG_master_pooled %>%
  filter(!is.na(w_fit), is.finite(w_fit), w_fit > 0)
m3_full <- glmmTMB::glmmTMB(
  formula = form_m1,
  data    = m3_full_data,
  weights = w_fit,
  family  = binomial(link = 'logit'),
  control = glmm_ctrl()
)
message('M3_full AIC: ', round(AIC(m3_full), 1))

# M2_full: all FG + all PAT + all eligible non-attempts (no train_game_ids filter)
non_filtered_full <- NON_master %>%
  filter(
    !is.na(p_hat_multinom) & p_hat_multinom >= NON_PI_THRESHOLD,
    !is.na(kick_distance)  & kick_distance  >  NON_DIST_THRESHOLD,
    kick_distance <= 70
  ) %>%
  mutate(kick_made = 0L, aug_weight = NON_WEIGHT)

pat_subset_full <- PAT_master %>%
  mutate(
    kick_made  = if_else(!is.na(kick_result), as.integer(kick_result == 'made'), NA_integer_),
    aug_weight = PAT_WEIGHT
  ) %>%
  filter(!is.na(kick_made))

train_m2_full <- dplyr::bind_rows(
  FG_master_pooled %>% mutate(aug_weight = 1.0),
  pat_subset_full,
  non_filtered_full
) %>%
  mutate(
    kicker_player_id = if_else(kicker_player_id %in% all_valid_kickers,
                               kicker_player_id, 'POOL'),
    stadium_id = if_else(stadium_id %in% all_valid_stadiums,
                         stadium_id, 'POOL')
  ) %>%
  filter(!is.na(kick_made), !is.na(aug_weight), aug_weight > 0)

message('Fitting M2_full ...')
m2_full <- glmmTMB::glmmTMB(
  formula = form_m1,
  data    = train_m2_full,
  weights = aug_weight,
  family  = binomial(link = 'logit'),
  control = glmm_ctrl()
)
message('M2_full AIC: ', round(AIC(m2_full), 1))

# M3-pop refit, full in-sample (companion to variant B below)
message('Fitting M3_pop_full (refit variant, full in-sample) ...')
m3_pop_full <- glmmTMB::glmmTMB(
  formula = form_m3_pop,
  data    = m3_full_data,
  weights = w_fit,
  family  = binomial(link = 'logit'),
  control = glmm_ctrl()
)
message('M3_pop_full AIC: ', round(AIC(m3_pop_full), 1))

# M2 no-PAT, full in-sample
message('Fitting M2_full (no PAT) ...')
train_m2_full_nopat <- dplyr::bind_rows(
  FG_master_pooled %>% mutate(aug_weight = 1.0),
  non_filtered_full
) %>%
  mutate(
    kicker_player_id = if_else(kicker_player_id %in% all_valid_kickers,
                               kicker_player_id, 'POOL'),
    stadium_id = if_else(stadium_id %in% all_valid_stadiums,
                         stadium_id, 'POOL')
  ) %>%
  filter(!is.na(kick_made), !is.na(aug_weight), aug_weight > 0)

m2_full_nopat <- glmmTMB::glmmTMB(
  formula = form_m1,
  data    = train_m2_full_nopat,
  weights = aug_weight,
  family  = binomial(link = 'logit'),
  control = glmm_ctrl()
)
message('M2_full (no PAT) AIC: ', round(AIC(m2_full_nopat), 1))
message('M2_full (no PAT) rows: ', nrow(train_m2_full_nopat))
message('M2_full rows: ', nrow(train_m2_full),
        ' (FG: ', nrow(FG_master_pooled), ', PAT: ', nrow(pat_subset_full),
        ', NON: ', nrow(non_filtered_full), ')')
message('In-sample model fits complete.')


Building in-sample pooled FG_master ...



Fitting M0_full ...



M0_full AIC: 7839.6



Fitting M1_full ...



M1_full AIC: 7811.1



...

Fitting M2_full (no PAT) ...



M2_full (no PAT) AIC: 8417.4



M2_full (no PAT) rows: 13471



M2_full rows: 27544 (FG: 11548, PAT: 14073, NON: 1923)



In-sample model fits complete.



In [16]:
# ============================================================
# 9b. In-Sample Predictions & Metrics
# ============================================================
message('Generating in-sample predictions ...')
fg_is_pred <- FG_master_pooled %>% mutate(aug_weight = 1.0)
FG_master_pooled <- FG_master_pooled %>%
  mutate(
    p_m0_is = predict_safe(m0_full, newdata = FG_master_pooled),
    p_m1_is = predict_safe(m1_full, newdata = FG_master_pooled),
    p_m3_is = predict_safe(m3_full, newdata = FG_master_pooled),
    p_m2_is = predict_safe(m2_full, newdata = fg_is_pred),
    p_m2_nopat_is       = predict_safe(m2_full_nopat, newdata = fg_is_pred),
    p_m3_pop_refit_is   = predict_safe(m3_pop_full, newdata = FG_master_pooled),
    p_m3_pop_zeroed_is  = predict_kicker_zeroed(m3_full, FG_master_pooled)
  )

# M2 must be evaluated only on actual FG kicks
fg_is_eval <- FG_master_pooled %>%
  dplyr::filter(is_pat == 0L, !is.na(kick_made))

message('In-sample FG rows for evaluation: ', nrow(fg_is_eval))
message('In-sample M2 non-NA predictions on FG rows: ',
        sum(!is.na(fg_is_eval$p_m2_is)), ' / ', nrow(fg_is_eval))

y_is <- fg_is_eval$kick_made

# --- Global ---
metrics_is_global <- dplyr::bind_rows(
  calc_metrics(y_is, fg_is_eval$p_m0_is, 'M0 (distance only)'),
  calc_metrics(y_is, fg_is_eval$p_m1_is, 'M1 (full GLMM)'),
  calc_metrics(y_is, fg_is_eval$p_m2_is, 'M2 (augmented; FG eval)'),
  calc_metrics(y_is, fg_is_eval$p_m3_is, 'M3 (IPW-corrected)'),
  calc_metrics(y_is, fg_is_eval$p_m2_nopat_is, 'M2 (no PAT; FG eval)'),
  calc_metrics(y_is, fg_is_eval$p_m3_pop_refit_is,  'M3-pop A (refit)'),
  calc_metrics(y_is, fg_is_eval$p_m3_pop_zeroed_is, 'M3-pop B (kicker zeroed)')
) %>% dplyr::mutate(subset = 'global')

# --- 50+ yards ---
fg_50_is <- fg_is_eval %>% dplyr::filter(kick_distance >= 50)
metrics_is_50 <- dplyr::bind_rows(
  calc_metrics(fg_50_is$kick_made, fg_50_is$p_m0_is, 'M0 (distance only)'),
  calc_metrics(fg_50_is$kick_made, fg_50_is$p_m1_is, 'M1 (full GLMM)'),
  calc_metrics(fg_50_is$kick_made, fg_50_is$p_m2_is, 'M2 (augmented; FG eval)'),
  calc_metrics(fg_50_is$kick_made, fg_50_is$p_m3_is, 'M3 (IPW-corrected)')
) %>% dplyr::mutate(subset = '50+ yards')

# --- 25-75% predicted probability zone (M1-defined) ---
fg_zone_is <- fg_is_eval %>% dplyr::filter(p_m1_is >= 0.25 & p_m1_is <= 0.75)
metrics_is_zone <- dplyr::bind_rows(
  calc_metrics(fg_zone_is$kick_made, fg_zone_is$p_m0_is, 'M0 (distance only)'),
  calc_metrics(fg_zone_is$kick_made, fg_zone_is$p_m1_is, 'M1 (full GLMM)'),
  calc_metrics(fg_zone_is$kick_made, fg_zone_is$p_m2_is, 'M2 (augmented; FG eval)'),
  calc_metrics(fg_zone_is$kick_made, fg_zone_is$p_m3_is, 'M3 (IPW-corrected)')
) %>% dplyr::mutate(subset = 'zone 25-75%')

metrics_insample <- dplyr::bind_rows(metrics_is_global, metrics_is_50, metrics_is_zone)

cat('\n=== In-Sample Metrics (Full Data) ===\n')
print(metrics_insample %>% dplyr::select(subset, set, n, brier, logloss, auc, calib_err))


Generating in-sample predictions ...



In-sample FG rows for evaluation: 11548



In-sample M2 non-NA predictions on FG rows: 11548 / 11548




=== In-Sample Metrics (Full Data) ===


# A tibble: 15 × 7
   subset      set                          n  brier logloss   auc calib_err
   <chr>       <chr>                    <int>  <dbl>   <dbl> <dbl>     <dbl>
 1 global      M0 (distance only)       11548 0.104    0.339 0.774   0.00577
 2 global      M1 (full GLMM)           11548 0.100    0.326 0.804   0.0136 
 3 global      M2 (augmented; FG eval)  11548 0.101    0.328 0.800   0.0190 
 4 global      M3 (IPW-corrected)       11548 0.0982   0.322 0.805   0.00823
 5 global      M2 (no PAT; FG eval)     11548 0.101    0.327 0.803   0.0182 
 6 global      M3-pop A (refit)         11548 0.104    0.336 0.779   0.00652
 7 global      M3-pop B (kicker zeroed) 11548 0.104    0.337 0.779   0.0100 
 8 50+ yards   M0 (distance only)        2129 0.210    0.610 0.589  NA      
 9 50+ yards   M1 (full GLMM)            2129 0.199    0.584 0.689   0.0570 
10 50+ yards   M2 (augmented; FG eval)   2129 0.200    0.588 0.684   0.0574 
11 50+ yards   M3 (IPW-corrected)        2129 0.187    0.

In [17]:
# ============================================================
# 9c. Calibration by Distance Range (In-Sample)
# ============================================================
# Bands are half-open on the right: [0,30), [30,40), ..., [60, Inf), so a
# label like '60+' means >= 60, which is what it reads as. This is the single
# band convention for the project: notebook 05, scripts/audit_paper_numbers.R
# and Presentation/make_slide_figures.R all cut the same way.
dist_breaks_is <- c(-Inf, 30, 40, 50, 60, Inf)
dist_labels_is <- c('<30', '30-39', '40-49', '50-59', '60+')

calib_range_insample <- FG_master_pooled %>%
  dplyr::mutate(
    dist_band = cut(kick_distance, breaks = dist_breaks_is,
                   labels = dist_labels_is, right = FALSE,
                   include.lowest = TRUE)
  ) %>%
  tidyr::pivot_longer(
    cols      = c(p_m0_is, p_m1_is, p_m2_is, p_m3_is),
    names_to  = 'model',
    values_to = 'p'
  ) %>%
  dplyr::mutate(model = dplyr::recode(model,
    'p_m0_is' = 'M0', 'p_m1_is' = 'M1', 'p_m2_is' = 'M2', 'p_m3_is' = 'M3'
  )) %>%
  dplyr::group_by(model, dist_band) %>%
  dplyr::summarise(
    n             = dplyr::n(),
    obs_make_pct  = mean(kick_made, na.rm = TRUE),
    pred_make_pct = mean(p, na.rm = TRUE),
    bias          = mean(p - kick_made, na.rm = TRUE),
    brier         = mean((kick_made - p)^2, na.rm = TRUE),
    .groups       = 'drop'
  )

cat('\n=== Calibration by Distance Range (In-Sample) ===\n')
print(calib_range_insample %>% dplyr::arrange(model, dist_band))

readr::write_csv(calib_range_insample,
                 file.path(reports_out, 'calibration_by_range_insample.csv'))
message('Saved: calibration_by_range_insample.csv')



=== Calibration by Distance Range (In-Sample) ===


# A tibble: 20 × 7
   model dist_band     n obs_make_pct pred_make_pct      bias  brier
   <chr> <fct>     <int>        <dbl>         <dbl>     <dbl>  <dbl>
 1 M0    <30        2709        0.983         0.985  0.00146  0.0162
 2 M0    30-39      3298        0.941         0.938 -0.00246  0.0553
 3 M0    40-49      3412        0.803         0.807  0.00359  0.156 
 4 M0    50-59      2033        0.697         0.693 -0.00358  0.209 
 5 M0    60+          96        0.406         0.398 -0.00858  0.230 
 6 M1    <30        2709        0.983         0.985  0.00193  0.0161
 7 M1    30-39      3298        0.941         0.939 -0.00115  0.0543
 8 M1    40-49      3412        0.803         0.809  0.00586  0.150 
 9 M1    50-59      2033        0.697         0.695 -0.00164  0.198 
10 M1    60+          96        0.406         0.391 -0.0149   0.219 
11 M2    <30        2709        0.983         0.985  0.00187  0.0161
12 M2    30-39      3298        0.941         0.928 -0.0122   0.0545
13 M2    40-49 

Saved: calibration_by_range_insample.csv



## 9. Evaluation & Predictions

In [18]:
# Generate test predictions from all models
test_m2 <- test_m1 %>%
  mutate(
    kicker_player_id = if_else(kicker_player_id %in% all_valid_kickers,
                               kicker_player_id, 'POOL'),
    stadium_id = if_else(stadium_id %in% all_valid_stadiums,
                         stadium_id, 'POOL'),
    aug_weight = 1.0
  )

message('Prediction row diagnostics:')
message('  test_df rows:  ', nrow(test_df))
message('  test_m1 rows:  ', nrow(test_m1))
message('  test_m2 rows:  ', nrow(test_m2))

preds_test <- test_df %>%
  select(game_id, play_id, season, kick_distance, kick_made, kick_result,
         kicker_player_id, stadium_id, is_pat, w_ipw_final, w_fit, weight_multinom_hajek) %>%
  mutate(
    p_m0 = predict_safe(m0, newdata = test_df),
    p_m1 = predict_safe(m1, newdata = test_m1),
    p_m3 = predict_safe(m3, newdata = test_m1),
    p_m2 = predict_safe(m2, newdata = test_m2),
    # M2 without PAT augmentation (plan 2.5)
    p_m2_nopat = predict_safe(m2_nopat, newdata = test_m2),
    # Two population-marginal baselines for FGOE (plan 2.4 / reviewer Main-5):
    #   A = separate refit without the kicker RE
    #   B = fitted M3 with the kicker RE zeroed, stadium retained
    p_m3_pop_refit  = predict_safe(m3_pop, newdata = test_m1),
    p_m3_pop_zeroed = predict_kicker_zeroed(m3, test_m1)
  )

# Full-set predictions (train + test) for output file
# Step 1: add pool helper columns
preds_full <- FG_master %>%
  mutate(
    kicker_player_id_pool = if_else(kicker_player_id %in% all_valid_kickers,
                                    kicker_player_id, 'POOL'),
    stadium_id_pool = if_else(stadium_id %in% all_valid_stadiums, stadium_id, 'POOL')
  )

# Step 2: pooled copy for RE models (kicker + stadium swapped to pool labels)
preds_full_pooled <- preds_full %>%
  mutate(
    kicker_player_id = kicker_player_id_pool,
    stadium_id       = stadium_id_pool,
    aug_weight       = 1.0
  )

# Step 3: generate predictions using explicit newdata objects, then drop helpers
preds_full <- preds_full %>%
  mutate(
    p_m0 = predict_safe(m0, newdata = preds_full),
    p_m1 = predict_safe(m1, newdata = preds_full_pooled),
    p_m3 = predict_safe(m3, newdata = preds_full_pooled),
    p_m2 = predict_safe(m2, newdata = preds_full_pooled),
    # Train-fitted variant predictions on every row, so notebook 04 can score
    # them on the held-out split without refitting or re-predicting.
    p_m2_nopat      = predict_safe(m2_nopat, newdata = preds_full_pooled),
    p_m3_pop_refit  = predict_safe(m3_pop,   newdata = preds_full_pooled),
    p_m3_pop_zeroed = predict_kicker_zeroed(m3, preds_full_pooled),
    p_m0_full = predict_safe(m0_full, newdata = preds_full),
    p_m1_full = predict_safe(m1_full, newdata = preds_full_pooled),
    p_m3_full = predict_safe(m3_full, newdata = preds_full_pooled),
    p_m2_full = predict_safe(m2_full, newdata = preds_full_pooled),
    p_m2_nopat_full = predict_safe(m2_full_nopat, newdata = preds_full_pooled),
    p_m3_pop_refit_full  = predict_safe(m3_pop_full, newdata = preds_full_pooled),
    p_m3_pop_zeroed_full = predict_kicker_zeroed(m3_full, preds_full_pooled)
  ) %>%
  select(-kicker_player_id_pool, -stadium_id_pool)

message('Predictions generated.')


Prediction row diagnostics:



  test_df rows:  2251



  test_m1 rows:  2251



  test_m2 rows:  2251



Predictions generated.



In [19]:
# ============================================================
# 10. Metrics Summary
# ============================================================
y_test <- preds_test$kick_made
m2_test_idx <- which(preds_test$is_pat == 0L & !is.na(preds_test$kick_made))

message('Test FG rows for M2 evaluation: ', length(m2_test_idx))
message('Test M2 non-NA predictions on FG rows: ',
        sum(!is.na(preds_test$p_m2[m2_test_idx])), ' / ', length(m2_test_idx))

metrics_all <- dplyr::bind_rows(
  calc_metrics(y_test, preds_test$p_m0, 'test_m0'),
  calc_metrics(y_test, preds_test$p_m1, 'test_m1'),
  calc_metrics(y_test, preds_test$p_m3, 'test_m3'),
  calc_metrics(preds_test$kick_made[m2_test_idx], preds_test$p_m2[m2_test_idx], 'test_m2_fg_only'),
  calc_metrics(preds_test$kick_made[m2_test_idx], preds_test$p_m2_nopat[m2_test_idx], 'test_m2_nopat_fg_only'),
  calc_metrics(y_test, preds_test$p_m3_pop_refit,  'test_m3_pop_refit'),
  calc_metrics(y_test, preds_test$p_m3_pop_zeroed, 'test_m3_pop_zeroed')
)

# Also compute on full set (train + test for reporting)
y_full <- preds_full$kick_made
m2_full_idx <- which(preds_full$is_pat == 0L & !is.na(preds_full$kick_made))
metrics_full <- dplyr::bind_rows(
  calc_metrics(y_full, preds_full$p_m0_full, 'full_m0'),
  calc_metrics(y_full, preds_full$p_m1_full, 'full_m1'),
  calc_metrics(y_full, preds_full$p_m3_full, 'full_m3'),
  calc_metrics(preds_full$kick_made[m2_full_idx], preds_full$p_m2_full[m2_full_idx], 'full_m2_fg_only'),
  # Section 6.2 compares M2 with and without the PAT block on BOTH splits; the
  # in-sample side of that comparison was previously computed but never exported,
  # which left the paper quoting numbers no artifact could verify.
  calc_metrics(preds_full$kick_made[m2_full_idx], preds_full$p_m2_nopat_full[m2_full_idx], 'full_m2_nopat_fg_only')
)

metrics_summary <- dplyr::bind_rows(metrics_all, metrics_full)

cat('\n=== Test Set Metrics ===\n')
print(metrics_all %>% dplyr::select(set, n, brier, logloss, auc, calib_err))

cat('\n=== In-Sample Metrics (Full Data) ===\n')
print(metrics_insample %>% dplyr::select(subset, set, n, brier, logloss, auc, calib_err))


Test FG rows for M2 evaluation: 2251



Test M2 non-NA predictions on FG rows: 2251 / 2251




=== Test Set Metrics ===


# A tibble: 7 × 6
  set                       n brier logloss   auc calib_err
  <chr>                 <int> <dbl>   <dbl> <dbl>     <dbl>
1 test_m0                2251 0.101   0.332 0.762    0.0189
2 test_m1                2251 0.100   0.330 0.769    0.0186
3 test_m3                2251 0.103   0.338 0.757    0.0315
4 test_m2_fg_only        2251 0.101   0.332 0.766    0.0253
5 test_m2_nopat_fg_only  2251 0.101   0.332 0.767    0.0224
6 test_m3_pop_refit      2251 0.101   0.333 0.764    0.0227
7 test_m3_pop_zeroed     2251 0.101   0.333 0.762    0.0208



=== In-Sample Metrics (Full Data) ===


# A tibble: 15 × 7
   subset      set                          n  brier logloss   auc calib_err
   <chr>       <chr>                    <int>  <dbl>   <dbl> <dbl>     <dbl>
 1 global      M0 (distance only)       11548 0.104    0.339 0.774   0.00577
 2 global      M1 (full GLMM)           11548 0.100    0.326 0.804   0.0136 
 3 global      M2 (augmented; FG eval)  11548 0.101    0.328 0.800   0.0190 
 4 global      M3 (IPW-corrected)       11548 0.0982   0.322 0.805   0.00823
 5 global      M2 (no PAT; FG eval)     11548 0.101    0.327 0.803   0.0182 
 6 global      M3-pop A (refit)         11548 0.104    0.336 0.779   0.00652
 7 global      M3-pop B (kicker zeroed) 11548 0.104    0.337 0.779   0.0100 
 8 50+ yards   M0 (distance only)        2129 0.210    0.610 0.589  NA      
 9 50+ yards   M1 (full GLMM)            2129 0.199    0.584 0.689   0.0570 
10 50+ yards   M2 (augmented; FG eval)   2129 0.200    0.588 0.684   0.0574 
11 50+ yards   M3 (IPW-corrected)        2129 0.187    0.

In [20]:
# ============================================================
# 10b. M3-pop variant comparison (plan 2.4 / reviewer Main-5)
# ============================================================
# The reviewer's specific worry: "I worry that not having the kicker in M3-pop
# will result in a bias for longer distances." A refit without the kicker RE has
# to re-absorb kicker ability into the distance basis, which can distort the
# long-distance shape; zeroing the RE out of the already-fitted M3 cannot. This
# cell tests exactly that, out of sample and by distance band.

m3pop_dist_breaks <- c(-Inf, 30, 40, 50, 60, Inf)
m3pop_dist_labels <- c('<30', '30-39', '40-49', '50-59', '60+')

m3pop_oos <- preds_test %>%
  filter(is_pat == 0L, !is.na(kick_made)) %>%
  mutate(dist_band = cut(kick_distance, breaks = m3pop_dist_breaks,
                         labels = m3pop_dist_labels, right = FALSE,
                         include.lowest = TRUE))

m3pop_global <- dplyr::bind_rows(
  calc_metrics(m3pop_oos$kick_made, m3pop_oos$p_m3_pop_refit,  'A_refit'),
  calc_metrics(m3pop_oos$kick_made, m3pop_oos$p_m3_pop_zeroed, 'B_kicker_zeroed')
)

cat('\n=== M3-pop variants: global OOS ===\n')
print(m3pop_global %>% dplyr::select(set, n, brier, logloss, auc, calib_err))

m3pop_by_band <- m3pop_oos %>%
  tidyr::pivot_longer(c(p_m3_pop_refit, p_m3_pop_zeroed),
                      names_to = 'variant', values_to = 'p') %>%
  mutate(variant = dplyr::recode(variant,
    'p_m3_pop_refit' = 'A_refit', 'p_m3_pop_zeroed' = 'B_kicker_zeroed')) %>%
  group_by(variant, dist_band) %>%
  summarise(
    n        = n(),
    obs      = mean(kick_made, na.rm = TRUE),
    pred     = mean(p, na.rm = TRUE),
    bias     = mean(p - kick_made, na.rm = TRUE),
    brier    = mean((kick_made - p)^2, na.rm = TRUE),
    logloss  = logloss(kick_made, p),
    .groups  = 'drop'
  )

cat('\n=== M3-pop variants: OOS calibration by distance band ===\n')
print(as.data.frame(m3pop_by_band), row.names = FALSE)

readr::write_csv(m3pop_global,  file.path(reports_out, 'm3pop_variant_global_oos.csv'))
readr::write_csv(m3pop_by_band, file.path(reports_out, 'm3pop_variant_by_distance_oos.csv'))
message('Saved: m3pop_variant_global_oos.csv, m3pop_variant_by_distance_oos.csv')

# Winner on OOS Brier decides which baseline FGOE uses downstream.
m3pop_winner <- m3pop_global$set[which.min(m3pop_global$brier)]
message('M3-pop variant with lower OOS Brier: ', m3pop_winner)



=== M3-pop variants: global OOS ===


# A tibble: 2 × 6
  set                 n brier logloss   auc calib_err
  <chr>           <int> <dbl>   <dbl> <dbl>     <dbl>
1 A_refit          2251 0.101   0.333 0.764    0.0227
2 B_kicker_zeroed  2251 0.101   0.333 0.762    0.0208



=== M3-pop variants: OOS calibration by distance band ===


         variant dist_band   n       obs      pred          bias      brier
         A_refit       <30 495 0.9818182 0.9849171  0.0030989534 0.01767987
         A_refit     30-39 644 0.9456522 0.9388644 -0.0067878193 0.05105064
         A_refit     40-49 696 0.8218391 0.8031579 -0.0186811486 0.14799865
         A_refit     50-59 397 0.7229219 0.6819220 -0.0409999333 0.19889501
         A_refit       60+  19 0.2631579 0.4610738  0.1979159235 0.24087461
 B_kicker_zeroed       <30 495 0.9818182 0.9867760  0.0049577935 0.01773060
 B_kicker_zeroed     30-39 644 0.9456522 0.9449666 -0.0006855928 0.05098702
 B_kicker_zeroed     40-49 696 0.8218391 0.8126922 -0.0091468856 0.14791768
 B_kicker_zeroed     50-59 397 0.7229219 0.6798716 -0.0430502705 0.19944348
 B_kicker_zeroed       60+  19 0.2631579 0.4247020  0.1615441075 0.22829477
    logloss
 0.08398766
 0.20475230
 0.47332886
 0.58659215
 0.67874817
 0.08504632
 0.20493185
 0.47348839
 0.58801826
 0.65638659


Saved: m3pop_variant_global_oos.csv, m3pop_variant_by_distance_oos.csv



M3-pop variant with lower OOS Brier: B_kicker_zeroed



In [21]:
# ============================================================
# 11. Save Outputs
# ============================================================
readr::write_csv(preds_full,      file.path(reports_out, 'fg_full_with_predictions.csv'))
readr::write_csv(metrics_summary, file.path(reports_out, 'metrics_summary.csv'))

message('\n=== Outputs Written ===')
message('fg_full_with_predictions.csv: ', nrow(preds_full), ' rows')
message('metrics_summary.csv:          ', nrow(metrics_summary), ' rows')
message('Models saved to:')
message('  models/final_models/', MODEL_FILE_M0)
message('  models/final_models/', MODEL_FILE_M1)
message('  models/final_models/', MODEL_FILE_M3)
message('  models/final_models/', MODEL_FILE_M3_POP)
message('  models/final_models/', MODEL_FILE_M2)


=== Outputs Written ===



fg_full_with_predictions.csv: 11548 rows



metrics_summary.csv:          12 rows



Models saved to:



  models/final_models/xfg_m0_dist_only_logit.rds



  models/final_models/xfg_m1_full_logit.rds



  models/final_models/xfg_m3_ipw_logit.rds



  models/final_models/xfg_m3_ipw_no_kicker_season_logit.rds



  models/final_models/xfg_m2_augmented_logit.rds



## 12. Year-by-Year Metrics Review

Per-season FG metrics to assess whether 2025 is uniquely less predictable relative to prior years.

In [22]:
# ============================================================
# 12. Year-by-Year Metrics Review
# ============================================================
metrics_by_season <- preds_full %>%
  filter(is_pat == 0L, !is.na(kick_made)) %>%
  group_by(season) %>%
  summarise(
    n          = n(),
    auc_m0     = auc_fn(kick_made, p_m0),
    auc_m1     = auc_fn(kick_made, p_m1),
    auc_m3     = auc_fn(kick_made, p_m3),
    auc_m2     = auc_fn(kick_made, p_m2),
    brier_m0   = brier(kick_made, p_m0),
    brier_m1   = brier(kick_made, p_m1),
    brier_m3   = brier(kick_made, p_m3),
    brier_m2   = brier(kick_made, p_m2),
    logloss_m0 = logloss(kick_made, p_m0),
    logloss_m1 = logloss(kick_made, p_m1),
    logloss_m3 = logloss(kick_made, p_m3),
    logloss_m2 = logloss(kick_made, p_m2),
    ess_m3     = ess(w_fit),
    ess_ratio_m3 = ess_m3 / n,
    .groups = 'drop'
  )

cat('\n=== Year-by-Year FG Metrics ===\n')
print(metrics_by_season)

readr::write_csv(metrics_by_season, file.path(reports_out, 'metrics_by_season.csv'))
message('Saved: metrics_by_season.csv')


=== Year-by-Year FG Metrics ===


# A tibble: 11 × 16
   season     n auc_m0 auc_m1 auc_m3 auc_m2 brier_m0 brier_m1 brier_m3 brier_m2
    <dbl> <int>  <dbl>  <dbl>  <dbl>  <dbl>    <dbl>    <dbl>    <dbl>    <dbl>
 1   2015  1009  0.790  0.797  0.797  0.793   0.0987   0.0977   0.0975   0.0984
 2   2016  1030  0.812  0.833  0.839  0.830   0.0995   0.0961   0.0935   0.0962
 3   2017  1042  0.721  0.748  0.743  0.744   0.111    0.107    0.108    0.108 
 4   2018   978  0.788  0.805  0.819  0.800   0.108    0.105    0.101    0.106 
 5   2019   999  0.794  0.809  0.803  0.805   0.120    0.116    0.113    0.116 
 6   2020  1002  0.759  0.780  0.800  0.774   0.108    0.106    0.101    0.106 
 7   2021  1060  0.791  0.803  0.808  0.800   0.101    0.0991   0.0970   0.0997
 8   2022  1077  0.763  0.784  0.774  0.782   0.0984   0.0955   0.0956   0.0967
 9   2023  1088  0.782  0.809  0.825  0.803   0.0981   0.0946   0.0915   0.0959
10   2024  1147  0.762  0.796  0.795  0.789   0.109    0.104    0.102    0.106 
11   2025  1116  0.7

Saved: metrics_by_season.csv



## 13. M2 Weight Sensitivity (Diagonal: PAT = NON)

Three-variant sensitivity over PAT and non-attempt weights with PAT=NON in {0.05, 0.10, 0.25}. Each variant trains on full in-sample FG+PAT+NON and is evaluated on FG-only in-sample predictions.

In [23]:
# ============================================================
# 13. M2 Weight Sensitivity (Diagonal: PAT = NON)
# ============================================================
m2_grid <- tibble::tibble(weight = c(0.05, 0.10, 0.25))
m2_grid_results <- vector('list', nrow(m2_grid))

# In-sample FG evaluation target (inferential focus)
y_is_fg <- FG_master_pooled$kick_made

# Pooling basis for full in-sample variants
all_valid_kickers_full  <- names(which(table(FG_master_pooled$kicker_player_id) >= 5))
all_valid_stadiums_full <- names(which(table(FG_master_pooled$stadium_id) >= 5))

for (i in seq_len(nrow(m2_grid))) {
  w <- m2_grid$weight[i]
  message(sprintf('M2 diagonal fit %d/%d with PAT=NON=%.2f', i, nrow(m2_grid), w))

  # Build full in-sample augmented training set: FG + PAT + NON
  stopifnot(max(NON_master$kick_distance, na.rm = TRUE) <= 70)
  non_filtered_w <- NON_master %>%
    filter(
      !is.na(p_hat_multinom) & p_hat_multinom >= NON_PI_THRESHOLD,
      !is.na(kick_distance)  & kick_distance  >  NON_DIST_THRESHOLD
    ) %>%
    mutate(
      kick_made = 0L,
      aug_weight = w,
      source_type = 'NON'
    )

  pat_subset_w <- PAT_master %>%
    mutate(
      kick_made = if_else(!is.na(kick_result), as.integer(kick_result == 'made'), NA_integer_),
      aug_weight = w,
      source_type = 'PAT'
    ) %>%
    filter(!is.na(kick_made))

  fg_subset_w <- FG_master_pooled %>%
    mutate(
      aug_weight = 1.0,
      source_type = 'FG'
    )

  train_m2_w <- dplyr::bind_rows(
    fg_subset_w,
    pat_subset_w,
    non_filtered_w
  ) %>%
    mutate(
      kicker_player_id = if_else(kicker_player_id %in% all_valid_kickers_full, kicker_player_id, 'POOL'),
      stadium_id       = if_else(stadium_id %in% all_valid_stadiums_full, stadium_id, 'POOL')
    ) %>%
    filter(!is.na(kick_made), !is.na(aug_weight), aug_weight > 0)

  source_diag <- train_m2_w %>%
    group_by(source_type) %>%
    summarise(
      n_rows = n(),
      total_weight = sum(aug_weight, na.rm = TRUE),
      .groups = 'drop'
    )

  fg_weight_total  <- dplyr::coalesce(source_diag$total_weight[source_diag$source_type == 'FG'][1], 0)
  pat_weight_total <- dplyr::coalesce(source_diag$total_weight[source_diag$source_type == 'PAT'][1], 0)
  non_weight_total <- dplyr::coalesce(source_diag$total_weight[source_diag$source_type == 'NON'][1], 0)

  fit_w <- tryCatch(
    glmmTMB::glmmTMB(
      formula = form_m1,
      data    = train_m2_w,
      weights = aug_weight,
      family  = binomial(link = 'logit'),
      control = glmm_ctrl()
    ),
    error = function(e) {
      message('Grid fit failed at weight=', w, ': ', e$message)
      NULL
    }
  )

  if (is.null(fit_w)) {
    m2_grid_results[[i]] <- tibble::tibble(
      pat_weight = w,
      non_weight = w,
      n_train    = nrow(train_m2_w),
      fg_weight_total  = fg_weight_total,
      pat_weight_total = pat_weight_total,
      non_weight_total = non_weight_total,
      pat_to_fg_weight = ifelse(fg_weight_total > 0, pat_weight_total / fg_weight_total, NA_real_),
      non_to_fg_weight = ifelse(fg_weight_total > 0, non_weight_total / fg_weight_total, NA_real_),
      coef_intercept   = NA_real_,
      coef_dist_bs_6   = NA_real_,
      auc_is_fg        = NA_real_,
      brier_is_fg      = NA_real_,
      logloss_is_fg    = NA_real_,
      calib_err_is_fg  = NA_real_
    )
    next
  }

  # In-sample prediction on FG rows only (newdata includes aug_weight per glmmTMB frame)
  fg_pred_w <- FG_master_pooled %>% mutate(aug_weight = 1.0)
  p_is_w <- predict_safe(fit_w, newdata = fg_pred_w)
  metrics_w <- calc_metrics(y_is_fg, p_is_w, sprintf('m2_is_w_%.2f', w))

  fixef_w <- tryCatch(glmmTMB::fixef(fit_w)$cond, error = function(e) numeric(0))
  coef_intercept <- if ('(Intercept)' %in% names(fixef_w)) unname(fixef_w['(Intercept)']) else NA_real_
  coef_dist_bs_6 <- if ('dist_bs_6' %in% names(fixef_w)) unname(fixef_w['dist_bs_6']) else NA_real_

  m2_grid_results[[i]] <- tibble::tibble(
    pat_weight = w,
    non_weight = w,
    n_train    = nrow(train_m2_w),
    fg_weight_total  = fg_weight_total,
    pat_weight_total = pat_weight_total,
    non_weight_total = non_weight_total,
    pat_to_fg_weight = ifelse(fg_weight_total > 0, pat_weight_total / fg_weight_total, NA_real_),
    non_to_fg_weight = ifelse(fg_weight_total > 0, non_weight_total / fg_weight_total, NA_real_),
    coef_intercept   = coef_intercept,
    coef_dist_bs_6   = coef_dist_bs_6,
    auc_is_fg        = metrics_w$auc,
    brier_is_fg      = metrics_w$brier,
    logloss_is_fg    = metrics_w$logloss,
    calib_err_is_fg  = metrics_w$calib_err
  )
}

m2_weight_grid <- dplyr::bind_rows(m2_grid_results) %>%
  arrange(pat_weight)

cat('\n=== M2 Diagonal Grid (In-Sample FG Evaluation) ===\n')
print(m2_weight_grid)

readr::write_csv(m2_weight_grid, file.path(reports_out, 'm2_weight_grid.csv'))
message('Saved: m2_weight_grid.csv')

M2 diagonal fit 1/3 with PAT=NON=0.05



Warning message in eval(family$initialize):
"non-integer #successes in a binomial glm!"


M2 diagonal fit 2/3 with PAT=NON=0.10



Warning message in eval(family$initialize):
"non-integer #successes in a binomial glm!"


M2 diagonal fit 3/3 with PAT=NON=0.25



Warning message in eval(family$initialize):
"non-integer #successes in a binomial glm!"


Warning message in finalizeTMB(TMBStruc, obj, fit, h, data.tmb.old):
"Model convergence problem; non-positive-definite Hessian matrix. See vignette('troubleshooting')"



=== M2 Diagonal Grid (In-Sample FG Evaluation) ===


# A tibble: 3 × 14
  pat_weight non_weight n_train fg_weight_total pat_weight_total
       <dbl>      <dbl>   <int>           <dbl>            <dbl>
1       0.05       0.05   27544           11548             704.
2       0.1        0.1    27544           11548            1407.
3       0.25       0.25   27544           11548            3518.
# ℹ 9 more variables: non_weight_total <dbl>, pat_to_fg_weight <dbl>,
#   non_to_fg_weight <dbl>, coef_intercept <dbl>, coef_dist_bs_6 <dbl>,
#   auc_is_fg <dbl>, brier_is_fg <dbl>, logloss_is_fg <dbl>,
#   calib_err_is_fg <dbl>


Saved: m2_weight_grid.csv



## 14. M2 pool experiments (diagnostic)

Not part of the reported models. Answers two questions about which non-attempts belong in M2's pseudo-miss pool; written up in `reports/m2_augmentation_experiments.md`.


In [24]:
# ============================================================
# 14. M2 pool experiments (diagnostic)
# ============================================================
# Written up in reports/m2_augmentation_experiments.md. Nothing here feeds the
# manuscript or the deck; these fits exist to answer two questions about WHICH
# non-attempts belong in M2's pseudo-miss pool.
#
# E1  Does lowering the propensity floor help? The floor selects situations where
#     kicking was plausible, and those are short kicks - where a Y=0 pseudo-miss
#     label is most wrong (a 30-39 yard kick is made 93% of the time, a 60+ yard
#     kick only 38%). Lowering the floor admits longer declines whose label is
#     less wrong, so if label quality is the binding constraint, a lower floor
#     should help rather than hurt.
#
# E2  Do the sub-33-yard declines carry information? They are currently excluded
#     on the assumption that a declined 30-yarder is a strategic choice carrying
#     no signal about the kick. Here they REPLACE the PAT block rather than
#     joining it, so the short-distance region is informed by declined fourth
#     downs instead of by extra points, without double-counting it.
#
# Every configuration is fitted on the TRAINING split and scored on both splits,
# so all rows are directly comparable to each other and to M2's primary numbers.
# The existing m2_nopat fit (pi >= 0.25, dist > 33, no PAT) is the reference that
# separates "removed the PAT block" from "added the sub-33 declines" in E2.

exp_grid <- tibble::tribble(
  ~config,                      ~pi_floor, ~dist_floor, ~use_pat,
  'E1 pi>=0.10',                     0.10,          33,     TRUE,
  'E1 pi>=0.15',                     0.15,          33,     TRUE,
  'M2 primary (pi>=0.25)',           0.25,          33,     TRUE,
  'E1 pi>=0.35',                     0.35,          33,     TRUE,
  'E1 pi>=0.50',                     0.50,          33,     TRUE,
  'E2 no PAT, no distance floor',    0.25,           0,    FALSE
)

# glmmTMB resolves its `weights` expression against newdata at predict time, so
# any frame passed to an augmented fit must carry an aug_weight column even
# though the value is irrelevant to a fitted-value prediction.
train_eval <- train_m1 %>% mutate(aug_weight = 1)
test_eval  <- test_m1  %>% mutate(aug_weight = 1)

y_train_fg <- train_eval$kick_made
y_test_fg  <- test_eval$kick_made

# Same half-open convention as the reported bands, so a bias quoted from
# this diagnostic and a count quoted from tbl-dist-brier refer to the same
# kicks.
band_of <- function(d) cut(d, breaks = dist_breaks_is, labels = dist_labels_is,
                           right = FALSE, include.lowest = TRUE)

band_summary <- function(fit, label) {
  p <- predict_safe(fit, newdata = train_eval)
  stopifnot(any(!is.na(p)))
  tibble::tibble(
    config    = label,
    dist_band = band_of(train_eval$kick_distance),
    y         = train_eval$kick_made,
    p         = p
  ) %>%
    dplyr::group_by(config, dist_band) %>%
    dplyr::summarise(
      n     = dplyr::n(),
      obs   = mean(y, na.rm = TRUE),
      pred  = mean(p, na.rm = TRUE),
      bias  = mean(p - y, na.rm = TRUE),
      brier = mean((y - p)^2, na.rm = TRUE),
      .groups = 'drop'
    )
}

exp_rows  <- list()
exp_bands <- list()

# M1 baseline: no augmentation at all. Refit is unnecessary - m1 is already the
# training-split fit - so it is scored through the same code path as the rest.
exp_rows[['M1']] <- dplyr::bind_cols(
  tibble::tibble(config = 'M1 (no augmentation)', pi_floor = NA_real_,
                 dist_floor = NA_real_, use_pat = NA, n_non = 0L,
                 n_non_train = 0L, pd_hessian = isTRUE(m1$sdr$pdHess),
                 non_weight_total = 0, pat_weight_total = 0),
  calc_metrics(y_train_fg, predict_safe(m1, newdata = train_eval), 'train') %>%
    dplyr::select(brier_is = brier, logloss_is = logloss, auc_is = auc,
                  calib_is = calib_err),
  calc_metrics(y_test_fg, predict_safe(m1, newdata = test_eval), 'test') %>%
    dplyr::select(brier_oos = brier, logloss_oos = logloss, auc_oos = auc,
                  calib_oos = calib_err)
)
exp_bands[['M1']] <- band_summary(m1, 'M1 (no augmentation)')

for (i in seq_len(nrow(exp_grid))) {
  cfg <- exp_grid[i, ]
  message(sprintf('[%d/%d] %s', i, nrow(exp_grid), cfg$config))

  non_exp <- NON_master %>%
    filter(
      !is.na(p_hat_multinom), p_hat_multinom >= cfg$pi_floor,
      !is.na(kick_distance),  kick_distance  >  cfg$dist_floor,
      kick_distance <= 70
    ) %>%
    mutate(kick_made = 0L, aug_weight = NON_WEIGHT, source_type = 'NON')

  blocks <- list(train_m1 %>% mutate(aug_weight = 1.0, source_type = 'FG'))
  if (isTRUE(cfg$use_pat)) {
    blocks[[length(blocks) + 1]] <- PAT_master %>%
      mutate(
        kick_made  = if_else(!is.na(kick_result), as.integer(kick_result == 'made'), NA_integer_),
        aug_weight = PAT_WEIGHT,
        source_type = 'PAT'
      ) %>%
      filter(!is.na(kick_made), game_id %in% train_game_ids)
  }
  blocks[[length(blocks) + 1]] <- non_exp %>% filter(game_id %in% train_game_ids)

  train_exp <- dplyr::bind_rows(blocks) %>%
    mutate(
      kicker_player_id = if_else(kicker_player_id %in% all_valid_kickers,
                                 kicker_player_id, 'POOL'),
      stadium_id       = if_else(stadium_id %in% all_valid_stadiums,
                                 stadium_id, 'POOL')
    ) %>%
    filter(!is.na(kick_made), !is.na(aug_weight), aug_weight > 0)

  wt <- train_exp %>%
    group_by(source_type) %>%
    summarise(total = sum(aug_weight), .groups = 'drop')
  gw <- function(s) dplyr::coalesce(wt$total[wt$source_type == s][1], 0)

  fit <- tryCatch(
    glmmTMB::glmmTMB(formula = form_m1, data = train_exp, weights = aug_weight,
                     family = binomial(link = 'logit'), control = glmm_ctrl()),
    error = function(e) { message('  FAILED: ', e$message); NULL }
  )
  if (is.null(fit)) next
  message(sprintf('  pool n=%d | NON weight=%.0f | PAT weight=%.0f | converged=%s',
                  nrow(non_exp), gw('NON'), gw('PAT'),
                  !isTRUE(fit$fit$convergence != 0)))

  exp_rows[[cfg$config]] <- dplyr::bind_cols(
    tibble::tibble(config = cfg$config, pi_floor = cfg$pi_floor,
                   dist_floor = cfg$dist_floor, use_pat = cfg$use_pat,
                   n_non = nrow(non_exp),
                   n_non_train = sum(non_exp$game_id %in% train_game_ids),
                   pd_hessian = isTRUE(fit$sdr$pdHess),
                   non_weight_total = gw('NON'), pat_weight_total = gw('PAT')),
    calc_metrics(y_train_fg, predict_safe(fit, newdata = train_eval), 'train') %>%
      dplyr::select(brier_is = brier, logloss_is = logloss, auc_is = auc,
                    calib_is = calib_err),
    calc_metrics(y_test_fg, predict_safe(fit, newdata = test_eval), 'test') %>%
      dplyr::select(brier_oos = brier, logloss_oos = logloss, auc_oos = auc,
                    calib_oos = calib_err)
  )
  exp_bands[[cfg$config]] <- band_summary(fit, cfg$config)
}

# The already-fitted no-PAT variant at the primary thresholds. Including it lets
# E2 be decomposed: this row minus M2 primary is the PAT block's contribution,
# and E2 minus this row is what the sub-33 declines add.
exp_rows[['m2_nopat_ref']] <- dplyr::bind_cols(
  tibble::tibble(config = 'ref: no PAT, dist>33 (existing m2_nopat)',
                 pi_floor = NON_PI_THRESHOLD, dist_floor = NON_DIST_THRESHOLD,
                 use_pat = FALSE, n_non = NA_integer_, n_non_train = NA_integer_,
                 pd_hessian = isTRUE(m2_nopat$sdr$pdHess),
                 non_weight_total = NA_real_, pat_weight_total = 0),
  calc_metrics(y_train_fg, predict_safe(m2_nopat, newdata = train_eval), 'train') %>%
    dplyr::select(brier_is = brier, logloss_is = logloss, auc_is = auc,
                  calib_is = calib_err),
  calc_metrics(y_test_fg, predict_safe(m2_nopat, newdata = test_eval), 'test') %>%
    dplyr::select(brier_oos = brier, logloss_oos = logloss, auc_oos = auc,
                  calib_oos = calib_err)
)
exp_bands[['m2_nopat_ref']] <- band_summary(m2_nopat, 'ref: no PAT, dist>33 (existing m2_nopat)')

exp_results <- dplyr::bind_rows(exp_rows)
exp_by_band <- dplyr::bind_rows(exp_bands)

# Bias relative to M1, which is what isolates the pseudo-label effect by band.
m1_band <- exp_by_band %>%
  dplyr::filter(config == 'M1 (no augmentation)') %>%
  dplyr::select(dist_band, bias_m1 = bias)
exp_by_band <- exp_by_band %>%
  dplyr::left_join(m1_band, by = 'dist_band') %>%
  dplyr::mutate(bias_vs_m1 = bias - bias_m1)

# Composition of the candidate pool at each floor, so the memo can say what the
# extra rows actually are rather than just how many there are.
pool_composition <- NON_master %>%
  filter(!is.na(p_hat_multinom), !is.na(kick_distance), kick_distance <= 70) %>%
  mutate(band = cut(kick_distance, c(0, 33, 40, 50, 55, 60, 70), right = TRUE)) %>%
  tidyr::crossing(floor = c(0.10, 0.15, 0.25, 0.35, 0.50)) %>%
  filter(p_hat_multinom >= floor) %>%
  group_by(floor, band) %>%
  summarise(n = n(), .groups = 'drop') %>%
  tidyr::pivot_wider(names_from = band, values_from = n, values_fill = 0)

readr::write_csv(exp_results,     file.path(reports_out, 'm2_pool_experiments.csv'))
readr::write_csv(exp_by_band,     file.path(reports_out, 'm2_pool_experiments_by_band.csv'))
readr::write_csv(pool_composition, file.path(reports_out, 'm2_pool_composition.csv'))

cat('\n=== M2 pool experiments: global ===\n')
print(as.data.frame(exp_results %>%
  dplyr::select(config, n_non, n_non_train, pd_hessian,
                brier_is, logloss_is, calib_is, brier_oos, logloss_oos)),
  digits = 5)

cat('\n=== bias vs M1 by distance band (training split) ===\n')
print(as.data.frame(exp_by_band %>%
  dplyr::select(config, dist_band, n, bias_vs_m1) %>%
  tidyr::pivot_wider(names_from = dist_band, values_from = c(n, bias_vs_m1))),
  digits = 3)

cat('\n=== candidate pool composition by propensity floor ===\n')
print(as.data.frame(pool_composition))


[1/6] E1 pi>=0.10



Warning message in eval(family$initialize):
"non-integer #successes in a binomial glm!"


Warning message in finalizeTMB(TMBStruc, obj, fit, h, data.tmb.old):
"Model convergence problem; non-positive-definite Hessian matrix. See vignette('troubleshooting')"


  pool n=2903 | NON weight=226 | PAT weight=1096 | converged=TRUE



[2/6] E1 pi>=0.15



...

                                    config n_non n_non_train pd_hessian
1                     M1 (no augmentation)     0           0       TRUE
2                              E1 pi>=0.10  2903        2265      FALSE
3                              E1 pi>=0.15  2470        1925      FALSE
4                    M2 primary (pi>=0.25)  1923        1488       TRUE
5                              E1 pi>=0.35  1543        1199      FALSE
6                              E1 pi>=0.50  1087         847      FALSE
7             E2 no PAT, no distance floor  3126        2420       TRUE
8 ref: no PAT, dist>33 (existing m2_nopat)    NA          NA       TRUE
  brier_is logloss_is calib_is brier_oos logloss_oos
1  0.10176    0.32957 0.010677   0.10044     0.32951
2  0.10272    0.33281 0.015962   0.10179     0.33346
3  0.10269    0.33282 0.015446   0.10160     0.33303
4  0.10257    0.33247 0.016687   0.10132     0.33227
5  0.10255    0.33239 0.015096   0.10115     0.33186
6  0.10247    0.33207 0.014049   0


=== bias vs M1 by distance band (training split) ===


                                    config n_<30 n_30-39 n_40-49 n_50-59 n_60+
1                     M1 (no augmentation)  2214    2654    2716    1636    77
2                              E1 pi>=0.10  2214    2654    2716    1636    77
3                              E1 pi>=0.15  2214    2654    2716    1636    77
4                    M2 primary (pi>=0.25)  2214    2654    2716    1636    77
5                              E1 pi>=0.35  2214    2654    2716    1636    77
6                              E1 pi>=0.50  2214    2654    2716    1636    77
7             E2 no PAT, no distance floor  2214    2654    2716    1636    77
8 ref: no PAT, dist>33 (existing m2_nopat)  2214    2654    2716    1636    77
  bias_vs_m1_<30 bias_vs_m1_30-39 bias_vs_m1_40-49 bias_vs_m1_50-59
1       0.00e+00          0.00000           0.0000           0.0000
2      -8.96e-05         -0.01199          -0.0175          -0.0470
3      -1.17e-04         -0.01169          -0.0171          -0.0383
4      -1.53e-04 


=== candidate pool composition by propensity floor ===


  floor (0,33] (33,40] (40,50] (50,55] (55,60] (60,70]
1  0.10   1460     467     905     837     666      28
2  0.15   1362     459     840     736     424      11
3  0.25   1203     424     746     555     197       1
4  0.35   1035     377     657     395     113       1
5  0.50    783     302     521     213      50       1
